# **AI Agents for Social Science & Society: Week 7**

## **RL to Optimize AI Agents**
- **Instructor:** James Evans
- **Notebook Authors & TAs:** Bhargav Srinivasa Desikan, Shiyang, Avi (+Claude)

This notebook covers reinforcement learning from bandits to RLHF, GRPO, and multi-agent systems. It is structured around four themes:

| Theme | Module | Status |
|-------|--------|--------|
| Deep RL Foundations | Module 1: Bandits, MDP, MC, TD, Policy Gradients, Actor-Critic, DQN, PPO | Required |
| RL for Alignment | Module 2: RLHF Pipeline, Constitutional AI & Reward Explainability | Required |
| RL for Reasoning | Module 3: GRPO, DeepSeek-R1 & Societies of Thought | Pick one of {3, 4} |
| Multi-Agent RL | Module 4: Multi-Agent RL, Theory of Mind & Collaborative Rewards | Pick one of {3, 4} |
| Memo | Module 5: RL for AI Agents & Institutions | Required |

**Requirements:** Complete Modules 1, 2, and 5. Choose one of Modules 3 or 4. Total: 4 modules.

**Here are some good resources to start with if you're new to RL:**
- [OpenAI Spinning Up — RL Introduction](https://spinningup.openai.com/en/latest/spinningup/rl_intro.html)
- [PyTorch RL Tutorial (Mario)](https://pytorch.org/tutorials/intermediate/mario_rl_tutorial.html)
- [RL Book Workshops](https://rl-book.com/learn/)
- [TRL Documentation](https://huggingface.co/docs/trl/)
- [RL Terms Cheatsheet](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N)

In [ ]:
# Core installs for all modules
!pip install numpy matplotlib torch torchvision pandas scikit-learn -q
!pip install gymnasium stable-baselines3 -q

# Module 1: Bandits
!pip install banditsbook==0.1.1 -q

# Module 2-3: LLM alignment and GRPO
!pip install "trl[peft]" transformers datasets accelerate bitsandbytes -q
!pip install openai -q

---
# Module 1: Deep RL Foundations

This module consolidates all foundational RL into a single progressive narrative. We start with the simplest RL setting — a multi-armed bandit — and build up to PPO, the algorithm that powers RLHF (Module 2) and whose simplification gives us GRPO (Module 3).

---

## Section A: Bandits & Exploration

We begin with the simplest RL setting: the [multi-armed bandit](https://en.wikipedia.org/wiki/Multi-armed_bandit). You have $K$ slot machines (arms), each with an unknown reward distribution. At each step, you pull one arm and observe a reward. The goal: maximize total reward over $T$ steps.

The fundamental tradeoff is **exploration vs. exploitation**: do you pull the arm that's been best so far (exploit), or try others that might be better (explore)? This tradeoff reappears throughout RL and is central to RLHF — when fine-tuning a language model, how much should it deviate from the reference policy to find better responses?

We compare three exploration strategies:
- **$\epsilon$-Greedy:** Exploit with probability $1-\epsilon$, explore randomly with probability $\epsilon$
- **Annealing:** High $\epsilon$ initially (explore a lot), decay over time (exploit more)
- **UCB (Upper Confidence Bound):** Explore arms that haven't been tried much, exploit those that have high estimated reward. Formally: $a_t = \arg\max_a \left[\hat{Q}(a) + c\sqrt{\frac{\ln t}{N(a)}}\right]$

For more on bandits: [RL Book Ch. 2](https://rl-book.com/learn/), [OpenAI Spinning Up — Exploration](https://spinningup.openai.com/en/latest/spinningup/rl_intro.html).

In [ ]:
from arms.bernoulli import BernoulliArm
from algorithms.epsilon_greedy.standard import EpsilonGreedy
from algorithms.softmax.annealing import AnnealingSoftmax
from algorithms.ucb.ucb1 import UCB1
from testing_framework.tests import test_algorithm
import pandas as pd
import matplotlib.pyplot as plt
import random

random.seed(42)
arm0 = BernoulliArm(0.05)
arm1 = BernoulliArm(0.4)
arms = [arm0, arm1]
num_sims = 1000
horizon = 250

results = pd.DataFrame()
for epsilon in [0, 0.1, 0.5, 1]:
    algo = EpsilonGreedy(epsilon, [], [])
    sim_nums, times, chosen_arms, rewards, cumulative_rewards = test_algorithm(algo, arms, num_sims, horizon)
    index = pd.MultiIndex.from_arrays(
        [[epsilon] * num_sims * horizon, sim_nums, times],
        names=('epsilon', 'simulation', 'time'))
    df = pd.DataFrame(chosen_arms, index=index).groupby(level=[0, 2]).sum() / num_sims
    results = pd.concat([results, df])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
results.unstack(level=0).plot(ax=ax1, ylim=[0, 1])
ax1.set_ylabel("P(Optimal Action)")
ax1.set_xlabel("Steps")
ax1.set_title("ε-Greedy: Exploration vs Exploitation")

n_arms = len(arms)
algo1 = AnnealingSoftmax([], []); algo1.initialize(n_arms)
algo2 = EpsilonGreedy(0.05, [], [])
algo3 = UCB1([], []); algo3.initialize(n_arms)

df_compare = pd.DataFrame()
for name, algo in [("ε-Greedy", algo2), ("Annealing", algo1), ("UCB", algo3)]:
    sim_nums, times, chosen_arms, rewards, cumulative_rewards = test_algorithm(algo, arms, num_sims, horizon)
    index = pd.MultiIndex.from_arrays([sim_nums, times], names=('simulation', 'time'))
    df_arm = pd.DataFrame(chosen_arms, index=index, columns=[name]).groupby(level=1).sum() / num_sims
    df_compare = pd.concat([df_compare, df_arm], axis=1)

df_compare.plot(ax=ax2, ylim=[0, 1])
ax2.set_ylabel("P(Optimal Action)")
ax2.set_xlabel("Steps")
ax2.set_title("Exploration Strategies Compared")
plt.tight_layout()
plt.show()

## Section B: The RL Dictionary

Reinforcement learning is a framework in which a computational **agent** learns to take **actions** in an **environment** to maximize cumulative **reward**. The formalism is the **Markov Decision Process (MDP)**.

![MDP Diagram](https://drive.google.com/uc?id=1dPaV7kJ12lndPH9SxRgT-mX-7hVuDaUi)

### Core Concepts

| Term | Definition |
|------|-----------|
| **State** $s \in S$ | Current observation of the environment |
| **Action** $a \in A$ | Choice made by the agent |
| **Reward** $r \in \mathbb{R}$ | Scalar feedback from environment |
| **Policy** $\pi(a\|s)$ | Probability of taking action $a$ in state $s$ |
| **Trajectory** $(s_0, a_0, r_1, s_1, a_1, r_2, \ldots)$ | Sequence of states, actions, rewards |
| **Return** $G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1}$ | Discounted cumulative reward |
| **Value function** $V^\pi(s) = \mathbb{E}_\pi[G_t \| s_t = s]$ | Expected return from state $s$ under $\pi$ |
| **Action value** $Q^\pi(s,a) = \mathbb{E}_\pi[G_t \| s_t = s, a_t = a]$ | Expected return from state $s$, action $a$ |
| **Bellman equation** | $V^\pi(s) = \mathbb{E}[r + \gamma V^\pi(s')]$ — value of a state in terms of successor states |

### Advance Concepts

| Term | Definition | Module |
|------|-----------|--------|
| **Reward Model** | A learned function $r_\theta(x, y)$ that scores outputs, trained on human preferences | Module 2 |
| **RLHF** | Reinforcement Learning from Human Feedback — learn reward from preferences, optimize with RL | Module 2 |
| **RLAIF** | RL from AI Feedback — replace human labelers with AI critique (Constitutional AI) | Module 2 |
| **KL Penalty** | $\beta \cdot \text{KL}(\pi_\theta \| \pi_\text{ref})$ — prevents policy from diverging too far from reference | Module 2 |
| **PPO** | Proximal Policy Optimization — clipped surrogate objective for stable policy updates | Section G |
| **GRPO** | Group Relative Policy Optimization — PPO without critic, uses group statistics as baseline | Module 3 |
| **Process Reward** | Reward each reasoning step, not just the final answer | Module 3 |
| **Credit Assignment** | Determining which agent contributed to team success | Module 4 |
| **Theory of Mind** | Modeling other agents' intentions to predict their actions | Module 4 |

### RL Settings

**Model-free vs. Model-based:** Model-free agents learn from interaction alone. Model-based agents learn or use a model of the environment.

**On-policy vs. Off-policy:** On-policy agents learn from their own actions (SARSA). Off-policy agents learn from any data, including past experience (Q-learning).

**Value-based vs. Policy-based:** Value-based methods learn $Q(s,a)$ and act greedily. Policy-based methods directly optimize $\pi_\theta(a|s)$. Actor-Critic combines both.

For the full RL dictionary with worked examples: [OpenAI Spinning Up](https://spinningup.openai.com/en/latest/spinningup/rl_intro.html), [RL Book workshops](https://rl-book.com/learn/).

## Section C: Dynamic Programming & Monte Carlo Methods

**Dynamic Programming (DP)** solves the Bellman equation exactly when we know the environment model. It iterates over all states, computing values from successor states. This works for small MDPs but is intractable for large ones.

**Monte Carlo (MC)** methods don't require a model. Instead, they *sample* — run many episodes, observe returns, and average. The MC estimate of the action value function:

$$Q(s, a) \approx \frac{1}{N(s,a)} \sum_{i=1}^{N(s,a)} G_i(s, a)$$

where $G_i(s,a)$ is the return observed after taking action $a$ in state $s$ during the $i$-th visit. With enough samples, this converges to the true value.

We implement this on a GridWorld — a 5x5 grid where an agent moves (N/E/S/W) toward a goal, receiving -1 per step. MC randomly explores, accumulates returns, and gradually identifies which actions lead to faster goal-reaching. This is adapted from [Winder, *Reinforcement Learning*](https://rl-book.com/learn/).

In [ ]:
from collections import defaultdict, namedtuple
from enum import Enum
from typing import List
import random
from IPython.display import clear_output

Point = namedtuple('Point', ['x', 'y'])

class Direction(Enum):
    NORTH = "⬆"; EAST = "⮕"; SOUTH = "⬇"; WEST = "⬅"
    @classmethod
    def values(cls): return [v for v in cls]

class SimpleGridWorld:
    def __init__(self, width=5, height=5):
        self.width, self.height = width, height
        self.action_space = [d for d in Direction]
        self.reset()

    def reset(self):
        self.cur_pos = Point(0, self.height - 1)
        self.goal = Point(self.width - 1, 0)
        return self.cur_pos, 0, False

    def step(self, action):
        x, y = self.cur_pos
        if action == Direction.NORTH: y += 1
        elif action == Direction.EAST: x += 1
        elif action == Direction.SOUTH: y -= 1
        elif action == Direction.WEST: x -= 1
        x, y = max(0, min(x, self.width - 1)), max(0, min(y, self.height - 1))
        self.cur_pos = Point(x, y)
        return self.cur_pos, -1, self.cur_pos == self.goal

    def __repr__(self):
        res = ""
        for y in reversed(range(self.height)):
            for x in range(self.width):
                if self.goal == Point(x, y):
                    res += "@" if self.cur_pos == Point(x, y) else "o"
                elif self.cur_pos == Point(x, y): res += "x"
                else: res += "_"
            res += "\n"
        return res

class MonteCarloAgent:
    def __init__(self, env, max_steps=1000):
        self.env = env
        self.max_steps = max_steps
        self.values = defaultdict(float)
        self.counts = defaultdict(float)

    def action_value(self, state, action):
        key = (state, action)
        return self.values[key] / self.counts[key] if self.counts[key] > 0 else 0.0

    def run_episode(self):
        buffer = []
        state, _, _ = self.env.reset()
        for _ in range(self.max_steps):
            action = random.choice(self.env.action_space)
            next_state, reward, terminal = self.env.step(action)
            buffer.append((state, action, reward))
            state = next_state
            if terminal: break
        episode_reward = 0
        for state, action, reward in reversed(buffer):
            episode_reward += reward
            key = (state, action)
            self.values[key] += episode_reward
            self.counts[key] += 1

env = SimpleGridWorld()
agent = MonteCarloAgent(env)
for i in range(1000):
    agent.run_episode()

def state_value_2d(env, agent):
    res = ""
    for y in reversed(range(env.height)):
        for x in range(env.width):
            if env.goal == Point(x, y): res += "   @  "
            else:
                sv = sum(agent.action_value(Point(x, y), d) for d in env.action_space) / len(env.action_space)
                res += f'{sv:6.2f}'
            res += " | "
        res += "\n"
    return res

def policy_2d(env, agent):
    res = ""
    for y in reversed(range(env.height)):
        for x in range(env.width):
            if env.goal == Point(x, y): res += "@"
            else:
                vals = [agent.action_value(Point(x, y), d) for d in env.action_space]
                res += env.action_space[vals.index(max(vals))].value
            res += " | "
        res += "\n"
    return res

print("State Values (average expected return from each position):")
print(state_value_2d(env, agent))
print("Optimal Policy (greedy over action values):")
print(policy_2d(env, agent))

## Section D: TD Learning & Q-Learning

Monte Carlo waits until an episode ends to update values. **Temporal Difference (TD)** learning updates after every *step*, using the current reward plus an estimate of future value:

$$V(s_t) \leftarrow V(s_t) + \alpha \big[r_{t+1} + \gamma V(s_{t+1}) - V(s_t)\big]$$

The term in brackets is the **TD error** $\delta_t$ — the difference between the observed reward-plus-estimate and the current estimate. This bootstrapping idea is fundamental to modern RL.

**Q-Learning** ([Watkins, 1989](https://en.wikipedia.org/wiki/Q-learning)) applies TD to action values and is **off-policy** — it learns the optimal $Q^*$ regardless of the agent's current behavior:

$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \big[r_{t+1} + \gamma \max_a Q(s_{t+1}, a) - Q(s_t, a_t)\big]$$

The $\max_a$ is the key — it always targets the best action in the next state. Q-learning with a neural network approximation for $Q$ is **DQN** (Section F). Q-learning with tabular states is what we implement here, on the Mountain Car environment.

For more: [RL Book Ch. 6](https://rl-book.com/learn/), [Sutton & Barto Ch. 6](http://incompleteideas.net/book/the-book.html).

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

env = gym.make("MountainCar-v0")

LEARNING_RATE = 0.1
DISCOUNT = 0.95
EPISODES = 10000
DISCRETE_GRID_SIZE = [20, 20]
EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY = EPISODES // 2

buckets = (env.observation_space.high - env.observation_space.low) / DISCRETE_GRID_SIZE
q_table = np.random.uniform(low=-2, high=0, size=(DISCRETE_GRID_SIZE + [env.action_space.n]))

def discretize(state):
    discrete = (state - env.observation_space.low) / buckets
    return tuple(np.clip(discrete.astype(int), 0, np.array(DISCRETE_GRID_SIZE) - 1))

epsilon = EPSILON_START
rewards_per_episode = []
success_count = 0

for episode in range(EPISODES):
    state, _ = env.reset()
    discrete_state = discretize(state)
    done, total_reward = False, 0

    while not done:
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[discrete_state])

        new_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        new_discrete = discretize(new_state)

        if not terminated:
            max_future_q = np.max(q_table[new_discrete])
            current_q = q_table[discrete_state + (action,)]
            q_table[discrete_state + (action,)] = current_q + LEARNING_RATE * (reward + DISCOUNT * max_future_q - current_q)
        elif new_state[0] >= env.unwrapped.goal_position:
            q_table[discrete_state + (action,)] = 0
            success_count += 1

        discrete_state = new_discrete
        total_reward += reward

    rewards_per_episode.append(total_reward)
    epsilon = max(EPSILON_END, epsilon - (EPSILON_START - EPSILON_END) / EPSILON_DECAY)

    if (episode + 1) % 2000 == 0:
        avg = np.mean(rewards_per_episode[-2000:])
        print(f"Episode {episode+1}: avg reward = {avg:.1f}, successes = {success_count}, ε = {epsilon:.3f}")
        success_count = 0

env.close()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
window = 100
smoothed = np.convolve(rewards_per_episode, np.ones(window)/window, mode='valid')
ax1.plot(smoothed)
ax1.set_xlabel("Episode")
ax1.set_ylabel("Reward (smoothed)")
ax1.set_title("Q-Learning on Mountain Car")

best_actions = q_table.argmax(axis=2)
action_names = {0: "←", 1: "·", 2: "→"}
ax2.imshow(best_actions.T, cmap='coolwarm', aspect='auto')
ax2.set_xlabel("Position bin")
ax2.set_ylabel("Velocity bin")
ax2.set_title("Q-table Policy (← left, → right)")
plt.tight_layout()
plt.show()

## Section E: Policy Gradients & Actor-Critic

Value-based methods (Q-learning, DQN) learn $Q(s,a)$ and act greedily. **Policy gradient** methods directly optimize the policy $\pi_\theta(a|s)$ by gradient ascent on expected return:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\big[G_t \cdot \nabla_\theta \ln \pi_\theta(a_t|s_t)\big]$$

This is the **REINFORCE** algorithm ([Williams, 1992](https://link.springer.com/article/10.1007/BF00992696)). Intuitively: if an action led to high return $G_t$, increase its probability; if low return, decrease it. The gradient of the log-policy $\nabla \ln \pi$ gives the direction to push.

REINFORCE has high variance because $G_t$ is a noisy single-sample return. **Actor-Critic** methods reduce variance by using a learned value function $V_\phi(s)$ as a baseline:

$$\nabla_\theta J(\theta) = \mathbb{E}\big[(G_t - V_\phi(s_t)) \cdot \nabla_\theta \ln \pi_\theta(a_t|s_t)\big]$$

The **actor** $\pi_\theta$ chooses actions. The **critic** $V_\phi$ evaluates them. This is the architecture behind PPO (Section G) and, by extension, RLHF (Module 2).

We implement REINFORCE and Actor-Critic on CartPole. For the Keras Actor-Critic tutorial this is adapted from: [keras.io/examples/rl/actor_critic_cartpole](https://keras.io/examples/rl/actor_critic_cartpole/).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

class PolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden)
        self.fc2 = nn.Linear(hidden, action_dim)

    def forward(self, x):
        return F.softmax(self.fc2(F.relu(self.fc1(x))), dim=-1)

def reinforce(env_name="CartPole-v1", episodes=500, gamma=0.99, lr=0.01):
    env = gym.make(env_name)
    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    episode_rewards = []

    for ep in range(episodes):
        state, _ = env.reset()
        log_probs, rewards = [], []
        done = False
        while not done:
            probs = policy(torch.FloatTensor(state))
            action = torch.multinomial(probs, 1).item()
            log_probs.append(torch.log(probs[action]))
            state, reward, terminated, truncated, _ = env.step(action)
            rewards.append(reward)
            done = terminated or truncated

        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        loss = sum(-lp * R for lp, R in zip(log_probs, returns))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        episode_rewards.append(sum(rewards))

    env.close()
    return episode_rewards

class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.shared = nn.Linear(state_dim, hidden)
        self.actor = nn.Linear(hidden, action_dim)
        self.critic = nn.Linear(hidden, 1)

    def forward(self, x):
        h = F.relu(self.shared(x))
        return F.softmax(self.actor(h), dim=-1), self.critic(h)

def actor_critic(env_name="CartPole-v1", episodes=500, gamma=0.99, lr=0.01):
    env = gym.make(env_name)
    model = ActorCritic(env.observation_space.shape[0], env.action_space.n)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    episode_rewards = []

    for ep in range(episodes):
        state, _ = env.reset()
        log_probs, values, rewards = [], [], []
        done = False
        while not done:
            state_t = torch.FloatTensor(state)
            probs, value = model(state_t)
            action = torch.multinomial(probs, 1).item()
            log_probs.append(torch.log(probs[action]))
            values.append(value.squeeze())
            state, reward, terminated, truncated, _ = env.step(action)
            rewards.append(reward)
            done = terminated or truncated

        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns)
        values = torch.stack(values)
        advantages = returns - values.detach()

        actor_loss = sum(-lp * A for lp, A in zip(log_probs, advantages))
        critic_loss = F.mse_loss(values, returns)
        loss = actor_loss + 0.5 * critic_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        episode_rewards.append(sum(rewards))

    env.close()
    return episode_rewards

print("Training REINFORCE on CartPole...")
reinforce_rewards = reinforce(episodes=500)
print("Training Actor-Critic on CartPole...")
ac_rewards = actor_critic(episodes=500)

fig, ax = plt.subplots(figsize=(10, 5))
window = 20
ax.plot(np.convolve(reinforce_rewards, np.ones(window)/window, mode='valid'), label="REINFORCE", alpha=0.8)
ax.plot(np.convolve(ac_rewards, np.ones(window)/window, mode='valid'), label="Actor-Critic", alpha=0.8)
ax.set_xlabel("Episode")
ax.set_ylabel("Episode Reward")
ax.set_title("REINFORCE vs Actor-Critic on CartPole")
ax.legend()
ax.axhline(y=195, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Section F: Deep Q-Networks (DQN)

Tabular Q-learning stores one value per (state, action) pair. This breaks in environments with large or continuous state spaces — like Atari games with raw pixel input. [Mnih et al. (2015)](https://www.nature.com/articles/nature14236) solved this by approximating $Q$ with a deep neural network: $Q(s, a; \theta) \approx Q^*(s, a)$.

Two innovations made this stable:

1. **Experience replay:** Store transitions $(s, a, r, s')$ in a buffer and sample random minibatches for training. This breaks temporal correlation and improves sample efficiency.

2. **Target network:** Use a separate, slowly-updated copy of the Q-network to compute targets $r + \gamma \max_a Q(s', a; \theta^-)$. This prevents the moving-target problem.

The DQN loss is:

$$\mathcal{L}(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}}\big[\big(r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta)\big)^2\big]$$

DQN achieved human-level performance on 49 Atari games from raw pixels — a landmark result. Experience replay and target networks reappear in RLHF training (Module 2), where the replay buffer stores (prompt, response, reward) tuples and the reference model acts as a target.

We implement DQN on CartPole. For the original paper: [Mnih et al. 2015, *Human-level control through deep reinforcement learning*](https://www.nature.com/articles/nature14236).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random

class DQN(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )

    def forward(self, x):
        return self.net(x)

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32),
                np.array(next_states), np.array(dones, dtype=np.float32))

    def __len__(self):
        return len(self.buffer)

env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

q_net = DQN(state_dim, action_dim)
target_net = DQN(state_dim, action_dim)
target_net.load_state_dict(q_net.state_dict())

optimizer = torch.optim.Adam(q_net.parameters(), lr=1e-3)
replay = ReplayBuffer(capacity=10000)

gamma = 0.99
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995
batch_size = 64
target_update_freq = 10
episode_rewards = []

for episode in range(300):
    state, _ = env.reset()
    total_reward = 0
    done = False

    while not done:
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                q_values = q_net(torch.FloatTensor(state))
                action = q_values.argmax().item()

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        replay.push(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

        if len(replay) >= batch_size:
            states, actions, rewards, next_states, dones = replay.sample(batch_size)
            states_t = torch.FloatTensor(states)
            actions_t = torch.LongTensor(actions)
            rewards_t = torch.FloatTensor(rewards)
            next_states_t = torch.FloatTensor(next_states)
            dones_t = torch.FloatTensor(dones)

            current_q = q_net(states_t).gather(1, actions_t.unsqueeze(1)).squeeze()
            with torch.no_grad():
                next_q = target_net(next_states_t).max(1)[0]
                target_q = rewards_t + gamma * next_q * (1 - dones_t)

            loss = F.mse_loss(current_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    episode_rewards.append(total_reward)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if (episode + 1) % target_update_freq == 0:
        target_net.load_state_dict(q_net.state_dict())

    if (episode + 1) % 50 == 0:
        avg = np.mean(episode_rewards[-50:])
        print(f"Episode {episode+1}: avg reward = {avg:.1f}, ε = {epsilon:.3f}")

env.close()

fig, ax = plt.subplots(figsize=(10, 4))
window = 20
smoothed = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
ax.plot(smoothed)
ax.set_xlabel("Episode")
ax.set_ylabel("Episode Reward")
ax.set_title("DQN on CartPole (experience replay + target network)")
ax.axhline(y=195, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Section G: Proximal Policy Optimization (PPO)

Policy gradient methods (Section E) update the policy by taking gradient steps on $\mathbb{E}[G \cdot \nabla \ln \pi]$. The problem: large steps can catastrophically degrade performance. **PPO** ([Schulman et al. 2017](https://arxiv.org/abs/1707.06347)) solves this with a *clipped surrogate objective* that limits how far the new policy can deviate from the old one:

$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta) \hat{A}_t,\; \text{clip}\big(r_t(\theta), 1-\epsilon, 1+\epsilon\big) \hat{A}_t\right)\right]$$

where $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_\text{old}}(a_t|s_t)}$ is the probability ratio and $\hat{A}_t$ is the estimated advantage.

The clipping prevents the ratio from moving too far from 1 — if the new policy assigns much higher probability to an action than the old policy, the objective is clipped, preventing an overly aggressive update. This makes PPO remarkably stable across different hyperparameter settings.

**Why PPO matters for the rest of this tutorial:**
- **Module 2 (RLHF):** PPO is the RL algorithm in the InstructGPT pipeline. It optimizes the language model policy against the reward model.
- **Module 3 (GRPO):** GRPO simplifies PPO by removing the critic network. Instead of learning $V(s)$, it uses group statistics as the baseline.

We implement PPO on CartPole using `stable-baselines3` (3 lines!) and from scratch (~80 lines) to see the internals.

For more: [Spinning Up — PPO](https://spinningup.openai.com/en/latest/algorithms/ppo.html), [Schulman et al. 2017](https://arxiv.org/abs/1707.06347), [TRL PPOTrainer](https://huggingface.co/docs/trl/ppo_trainer).

In [ ]:
from stable_baselines3 import PPO
import gymnasium as gym
import numpy as np

model = PPO("MlpPolicy", "CartPole-v1", verbose=1, n_steps=256, n_epochs=10, batch_size=64)
model.learn(total_timesteps=50000)

env = gym.make("CartPole-v1")
rewards = []
for _ in range(100):
    obs, _ = env.reset()
    total_reward = 0
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        done = terminated or truncated
    rewards.append(total_reward)

env.close()
print(f"PPO (stable-baselines3): {np.mean(rewards):.1f} ± {np.std(rewards):.1f} reward over 100 episodes")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

class PPOActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=64):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, action_dim))
        self.critic = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1))

    def act(self, state):
        logits = self.actor(state)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        return action, dist.log_prob(action), self.critic(state).squeeze()

    def evaluate(self, states, actions):
        logits = self.actor(states)
        dist = torch.distributions.Categorical(logits=logits)
        return dist.log_prob(actions), dist.entropy(), self.critic(states).squeeze()

def collect_rollout(env, model, n_steps=2048):
    states, actions, rewards, log_probs, values, dones = [], [], [], [], [], []
    state, _ = env.reset()
    for _ in range(n_steps):
        state_t = torch.FloatTensor(state)
        with torch.no_grad():
            action, log_prob, value = model.act(state_t)
        next_state, reward, terminated, truncated, _ = env.step(action.item())
        states.append(state); actions.append(action.item()); rewards.append(reward)
        log_probs.append(log_prob.item()); values.append(value.item())
        dones.append(terminated or truncated)
        state = next_state if not (terminated or truncated) else env.reset()[0]
    return states, actions, rewards, log_probs, values, dones

def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    advantages, gae = [], 0
    for t in reversed(range(len(rewards))):
        next_value = values[t + 1] if t + 1 < len(values) else 0
        delta = rewards[t] + gamma * next_value * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages.insert(0, gae)
    returns = [a + v for a, v in zip(advantages, values)]
    return advantages, returns

env = gym.make("CartPole-v1")
model = PPOActorCritic(env.observation_space.shape[0], env.action_space.n)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
clip_eps = 0.2
n_epochs = 10
n_steps = 2048
total_updates = 25
all_rewards = []

for update in range(total_updates):
    states, actions, rewards, old_log_probs, values, dones = collect_rollout(env, model, n_steps)
    advantages, returns = compute_gae(rewards, values, dones)

    states_t = torch.FloatTensor(np.array(states))
    actions_t = torch.LongTensor(actions)
    old_log_probs_t = torch.FloatTensor(old_log_probs)
    advantages_t = torch.FloatTensor(advantages)
    advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)
    returns_t = torch.FloatTensor(returns)

    for epoch in range(n_epochs):
        new_log_probs, entropy, new_values = model.evaluate(states_t, actions_t)
        ratio = torch.exp(new_log_probs - old_log_probs_t)
        surr1 = ratio * advantages_t
        surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantages_t
        actor_loss = -torch.min(surr1, surr2).mean()
        critic_loss = F.mse_loss(new_values, returns_t)
        loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy.mean()

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()

    ep_rewards, ep_reward = [], 0
    for r, d in zip(rewards, dones):
        ep_reward += r
        if d:
            ep_rewards.append(ep_reward)
            ep_reward = 0
    all_rewards.extend(ep_rewards)
    print(f"Update {update+1}/{total_updates}: mean reward = {np.mean(ep_rewards):.1f}")

env.close()

fig, ax = plt.subplots(figsize=(10, 4))
window = 10
smoothed = np.convolve(all_rewards, np.ones(window)/window, mode='valid') if len(all_rewards) > window else all_rewards
ax.plot(smoothed)
ax.set_xlabel("Episode")
ax.set_ylabel("Episode Reward")
ax.set_title("PPO from Scratch on CartPole")
ax.axhline(y=195, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
# Module 2: RLHF Pipeline, Constitutional AI & Reward Explainability

In Module 1 we trained RL agents on environments with well-defined reward functions — CartPole gives +1 each timestep the pole stays up, Breakout gives points for breaking bricks. But what happens when the reward is not a number from an environment, but a *human preference*? This is the central problem of aligning large language models.

The insight of [Christiano et al. (2017)](https://arxiv.org/abs/1706.03741) was to learn a reward model from pairwise human comparisons, then optimize against it with RL. [Ouyang et al. (2022)](https://arxiv.org/abs/2203.02155) scaled this to GPT-3, producing InstructGPT — the template behind ChatGPT. [Bai et al. (2022)](https://arxiv.org/abs/2212.08073) replaced human labelers with AI feedback, creating Constitutional AI. And [Reber et al. (2025)](https://github.com/toddnief/RATE) asked: what are these reward models actually rewarding?

In this module we build the full pipeline: train a reward model, fine-tune a language model with PPO, implement a constitutional critique-revision loop, and measure what our reward model is sensitive to using causal methods.

![InstructGPT Pipeline](https://miro.medium.com/v2/resize:fit:1100/format:webp/1*WNjCnrKgMTsiEaexazCKaQ.png)
*Figure: The InstructGPT training pipeline (Ouyang et al. 2022). Step 1: SFT on demonstrations. Step 2: Train reward model on comparisons. Step 3: Optimize policy with PPO against the RM.*

---

## Section A: Reward Model from Human Preferences

The foundational insight of RLHF comes from [Christiano et al. (2017)](https://arxiv.org/abs/1706.03741): instead of hand-specifying a reward function, *learn* it from human comparisons. Given two completions $y_w$ and $y_l$ for the same prompt $x$, a human labels which is better. The reward model learns to assign higher scores to preferred completions.

The standard model is **Bradley-Terry**. Given a learned reward function $r_\theta$:

$$P(y_w \succ y_l) = \sigma\big(r_\theta(x, y_w) - r_\theta(x, y_l)\big)$$

The training loss maximizes the log-likelihood of observed preferences:

$$\mathcal{L}(\theta) = -\mathbb{E}_{(x, y_w, y_l)}\left[\log \sigma\big(r_\theta(x, y_w) - r_\theta(x, y_l)\big)\right]$$

This is what TRL's `RewardTrainer` computes internally. We train on [`trl-lib/ultrafeedback_binarized`](https://huggingface.co/datasets/trl-lib/ultrafeedback_binarized) — 61K preference pairs annotated by GPT-4. Each example has a `chosen` and `rejected` conversation. We use `Qwen/Qwen2.5-0.5B-Instruct` as backbone — small enough for a T4, capable enough to learn meaningful preferences.

For more on reward modeling: [TRL RewardTrainer docs](https://huggingface.co/docs/trl/reward_trainer), [Ouyang et al. (2022) §3.2](https://arxiv.org/abs/2203.02155).

In [ ]:
from datasets import load_dataset
from trl import RewardTrainer, RewardConfig

dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train[:5000]")
eval_dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="test[:500]")

print(f"Train: {len(dataset)} | Eval: {len(eval_dataset)}")
print(f"Sample chosen:\n{dataset[0]['chosen'][:2]}\n")
print(f"Sample rejected:\n{dataset[0]['rejected'][:2]}")

reward_config = RewardConfig(
    output_dir="reward_model",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    max_length=512,
    bf16=True,
)

reward_trainer = RewardTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=reward_config,
    train_dataset=dataset,
    eval_dataset=eval_dataset,
)

reward_trainer.train()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

rm_model = reward_trainer.model
rm_tokenizer = reward_trainer.tokenizer

def score_response(model, tokenizer, prompt, response):
    messages = [{"role": "user", "content": prompt}, {"role": "assistant", "content": response}]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", padding=True, truncation=True, max_length=512
    ).to(model.device)
    with torch.no_grad():
        return model(inputs).logits[0].item()

chosen_scores, rejected_scores = [], []
for ex in eval_dataset.select(range(100)):
    prompt = ex["chosen"][0]["content"]
    chosen_scores.append(score_response(rm_model, rm_tokenizer, prompt, ex["chosen"][-1]["content"]))
    rejected_scores.append(score_response(rm_model, rm_tokenizer, prompt, ex["rejected"][-1]["content"]))

accuracy = np.mean(np.array(chosen_scores) > np.array(rejected_scores))
print(f"Ranking accuracy: {accuracy:.1%}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(chosen_scores, bins=30, alpha=0.6, label=f"Chosen (mean={np.mean(chosen_scores):.2f})")
ax.hist(rejected_scores, bins=30, alpha=0.6, label=f"Rejected (mean={np.mean(rejected_scores):.2f})")
ax.set_xlabel("Reward Score")
ax.set_ylabel("Count")
ax.set_title(f"RM Score Distribution — Accuracy: {accuracy:.1%}")
ax.legend()
plt.tight_layout()
plt.show()

## Section B: PPO Fine-Tuning of a Language Model

Now we have a reward model that scores responses. The next step is to use it as a training signal. We load `Qwen/Qwen2.5-0.5B-Instruct` as both the **policy** (the model we're training) and the **reference** (a frozen copy). The training loop:

1. **Generate** a response from the policy: $y \sim \pi_\theta(y|x)$
2. **Score** with the reward model: $r = r_\phi(x, y)$
3. **Compute KL penalty** against the reference policy: $\text{KL} = \mathbb{E}[\log \pi_\theta(y|x) - \log \pi_\text{ref}(y|x)]$
4. **Compute penalized reward:** $R(x, y) = r_\phi(x, y) - \beta \cdot \text{KL}(\pi_\theta \| \pi_\text{ref})$
5. **Update** the policy to increase probability of high-reward responses

The KL penalty prevents the policy from drifting too far from the reference — without it, the model would quickly overfit to the reward model's quirks (reward hacking). This is the core of the InstructGPT pipeline ([Ouyang et al. 2022](https://arxiv.org/abs/2203.02155)). We implement a simplified REINFORCE-style update for clarity.

> **Going Further:** For production RLHF, use TRL's `PPOTrainer` or `OnlineDPOTrainer` which handle rollout buffers, minibatch updates, and distributed training. See [TRL PPO docs](https://huggingface.co/docs/trl/ppo_trainer).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import copy

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16).to(device)
ref_policy = copy.deepcopy(policy)
ref_policy.eval()
for p in ref_policy.parameters():
    p.requires_grad = False

optimizer = torch.optim.Adam(policy.parameters(), lr=1e-6)
beta = 0.1
max_new_tokens = 128

prompts = [
    "What makes a good research question in social science?",
    "How should AI systems be governed?",
    "Explain the concept of institutional design.",
    "What are the ethical implications of algorithmic decision-making?",
    "How do social norms emerge in organizations?",
    "What role does trust play in collaborative AI systems?",
    "How can we measure fairness in machine learning?",
    "What are the tradeoffs between efficiency and equity in policy design?",
]

reward_history = []
n_epochs = 3

for epoch in range(n_epochs):
    epoch_rewards = []
    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        inputs = tokenizer(text, return_tensors="pt").to(device)
        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            output_ids = policy.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
        response_ids = output_ids[0][input_len:]
        response_text = tokenizer.decode(response_ids, skip_special_tokens=True)

        rm_score = score_response(rm_model, rm_tokenizer, prompt, response_text)

        policy_logits = policy(output_ids).logits[0, input_len-1:-1]
        ref_logits = ref_policy(output_ids).logits[0, input_len-1:-1]
        policy_logprobs = torch.log_softmax(policy_logits, dim=-1)
        ref_logprobs = torch.log_softmax(ref_logits, dim=-1)
        token_policy_lp = policy_logprobs.gather(1, response_ids.unsqueeze(1)).squeeze()
        token_ref_lp = ref_logprobs.gather(1, response_ids.unsqueeze(1)).squeeze()
        kl = (token_policy_lp - token_ref_lp).mean()

        reward = rm_score - beta * kl.item()
        loss = -token_policy_lp.mean() * reward

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer.step()

        epoch_rewards.append(rm_score)
        reward_history.append(rm_score)

    print(f"Epoch {epoch+1}/{n_epochs}: mean RM score = {np.mean(epoch_rewards):.3f}, KL = {kl.item():.4f}")

In [ ]:
test_prompt = "What makes a good research question in social science?"
messages = [{"role": "user", "content": test_prompt}]
text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
inputs = tokenizer(text, return_tensors="pt").to(device)
input_len = inputs["input_ids"].shape[1]

with torch.no_grad():
    before_ids = ref_policy.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
before_text = tokenizer.decode(before_ids[0][input_len:], skip_special_tokens=True)
before_score = score_response(rm_model, rm_tokenizer, test_prompt, before_text)

with torch.no_grad():
    after_ids = policy.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
after_text = tokenizer.decode(after_ids[0][input_len:], skip_special_tokens=True)
after_score = score_response(rm_model, rm_tokenizer, test_prompt, after_text)

print(f"Before RLHF (RM score: {before_score:.3f}):\n{before_text[:500]}")
print(f"\nAfter RLHF (RM score: {after_score:.3f}):\n{after_text[:500]}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(reward_history)
ax1.set_xlabel("Step")
ax1.set_ylabel("RM Score")
ax1.set_title("RM Score During RLHF Training")
ax2.bar(["Before", "After"], [before_score, after_score], color=["#e74c3c", "#2ecc71"])
ax2.set_ylabel("RM Score")
ax2.set_title("Before vs After RLHF")
plt.tight_layout()
plt.show()

## Section C: Constitutional AI / RLAIF

RLHF depends on human labelers — expensive, slow, and hard to scale. [Bai et al. (2022)](https://arxiv.org/abs/2212.08073) proposed **Constitutional AI**: replace humans with a set of **principles** that an AI uses to critique and revise its own outputs. The process:

1. **Generate** a response to a prompt
2. **Critique** the response against a constitutional principle (e.g., "Is this response harmful?")
3. **Revise** the response to address the critique
4. **Score both** with the reward model: the revised version should score higher

The 16 critique-revision templates below are from [Bai et al. (2022)](https://arxiv.org/abs/2212.08073). Each encodes a normative commitment — harmful content, age-appropriateness, toxicity, criminal activity. These are institutional choices: who writes the constitution? Whose values does it encode?

We use `gpt-4o-mini` via the OpenAI API for the critique-revision loop. Students need an `OPENAI_API_KEY` environment variable set.

> **Going Further:** For a fully local implementation, swap the API call for a local model using `transformers.pipeline("text-generation")`. The [ConstitutionalHarmlessnessPaper repo](https://github.com/anthropics/ConstitutionalHarmlessnessPaper) has all 16 templates and the full implementation.

In [ ]:
from openai import OpenAI
import os

client = OpenAI()

CONSTITUTIONAL_PRINCIPLES = {
    "harmful": {
        "critique": "Identify specific ways in which the assistant's last response is harmful, unethical, racist, sexist, toxic, dangerous, or illegal.",
        "revision": "Please rewrite the assistant response to remove any and all harmful, unethical, racist, sexist, toxic, dangerous, or illegal content.",
    },
    "age_appropriate": {
        "critique": "Did the assistant's preceding response include anything that may be inappropriate for young children? If so, discuss how it can be revised to be more appropriate.",
        "revision": "Please rewrite the assistant response to make it more age-appropriate for children.",
    },
    "empathetic": {
        "critique": "Please comment on whether the assistant's last response is thoughtful and empathetic. Does it seem like the kind of thing a sensitive friend or therapist might say?",
        "revision": "Please write a new assistant response so that it is more like what a thoughtful, empathetic, caring, sensitive friend or therapist would say.",
    },
    "criminal": {
        "critique": "Read the assistant's last response carefully and identify anything it said that provides advice or assistance in potentially criminal activities.",
        "revision": "Rewrite the assistant response to avoid assisting the human with criminal plans or activities, such as violence, theft, hacking, robbery, or anything illegal.",
    },
}

def critique_and_revise(client, prompt, response, principle):
    critique = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that critiques responses."},
            {"role": "user", "content": f"Human: {prompt}\n\nAssistant: {response}\n\nCritiqueRequest: {principle['critique']}\n\nCritique:"},
        ],
        max_tokens=300,
    ).choices[0].message.content

    revision = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"Human: {prompt}\n\nAssistant: {response}\n\nCritique: {critique}\n\nRevisionRequest: {principle['revision']}\n\nRevision:"},
        ],
        max_tokens=300,
    ).choices[0].message.content

    return critique, revision

cai_prompts = [
    "What makes a good research question in social science?",
    "How should AI be used in criminal justice?",
    "Explain the effects of social media on democracy.",
]

cai_results = []
for prompt in cai_prompts:
    original = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
    ).choices[0].message.content

    original_score = score_response(rm_model, rm_tokenizer, prompt, original)
    print(f"\nPrompt: {prompt[:60]}...")
    print(f"  Original RM score: {original_score:.3f}")

    for principle_name, principle in CONSTITUTIONAL_PRINCIPLES.items():
        critique, revision = critique_and_revise(client, prompt, original, principle)
        revised_score = score_response(rm_model, rm_tokenizer, prompt, revision)
        delta = revised_score - original_score
        cai_results.append({"prompt": prompt, "principle": principle_name, "original": original_score, "revised": revised_score, "delta": delta})
        print(f"  {principle_name}: revised={revised_score:.3f} (delta={delta:+.3f})")

import pandas as pd
fig, ax = plt.subplots(figsize=(8, 5))
df_cai = pd.DataFrame(cai_results)
means = df_cai.groupby("principle")["delta"].mean()
sems = df_cai.groupby("principle")["delta"].sem()
means.plot.barh(xerr=sems, ax=ax, color="#2ecc71", alpha=0.8)
ax.set_xlabel("Mean RM Score Change (revised - original)")
ax.set_title("Constitutional AI: Effect of Critique-Revision on RM Score")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

## Section D: Reward Model Explainability — RATE

We have a trained reward model. But what is it actually rewarding? [Reber et al. (2025)](https://github.com/toddnief/RATE) introduce RATE (Rewrite-based Attribute Treatment Estimators) — a causal method to measure how specific text attributes affect RM scores.

The idea borrows from causal inference. To measure the causal effect of an attribute $W$ (say, verbosity) on the reward score $Y$:

1. **Generate** a response and **rewrite** it to add the attribute ($W\!=\!0 \to W\!=\!1$)
2. **Rewrite the rewrite** back to remove the attribute ($W\!=\!1 \to W\!=\!0$) — the *rewritten rewrite*
3. **Score all versions** with the RM
4. **Compute the Average Treatment Effect:** $\text{ATE} = \mathbb{E}[Y(W\!=\!1)] - \mathbb{E}[Y(W\!=\!0)]$

Why the double rewrite? A single rewrite might change confounding attributes (making text verbose might also add helpful details). The rewritten rewrite controls for confounds — same logic as [difference-in-differences](https://en.wikipedia.org/wiki/Difference_in_differences) in econometrics.

We use `gpt-4o-mini` for rewrites and the RM from Section A for scoring. We test four attributes: **verbosity**, **formality**, **humor**, and **helpfulness**.

> **Going Further:** The full RATE pipeline ([github.com/toddnief/RATE](https://github.com/toddnief/RATE)) supports batch processing via OpenAI's Batch API, multiple RMs including [ArmoRM-Llama3-8B](https://huggingface.co/RLHFlow/ArmoRM-Llama3-8B-v0.1), and more sophisticated estimators (ATT, ATU).

In [ ]:
def rewrite_text(client, text, instruction):
    return client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"{instruction}\n\nOriginal text:\n{text}"}],
        max_tokens=500,
    ).choices[0].message.content

def generate_response(client, prompt):
    return client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
    ).choices[0].message.content

attributes = {
    "verbose": {
        "add": "Rewrite to be significantly more verbose and detailed, keeping the core meaning.",
        "remove": "Rewrite to be more concise and to-the-point, keeping the core meaning.",
    },
    "formal": {
        "add": "Rewrite in a much more formal, academic tone, keeping the core meaning.",
        "remove": "Rewrite in a casual, conversational tone, keeping the core meaning.",
    },
    "humorous": {
        "add": "Rewrite to include humor and wit, keeping the core meaning.",
        "remove": "Rewrite to remove any humor, making it straightforward and serious.",
    },
    "helpful": {
        "add": "Rewrite to be more helpful and actionable with specific advice, keeping the core meaning.",
        "remove": "Rewrite to be more vague and general, removing specific advice.",
    },
}

sample_prompts = [
    "How should AI be regulated in healthcare?",
    "What are the effects of social media on democracy?",
    "How can organizations improve diversity and inclusion?",
    "What role should AI play in criminal justice?",
    "How do institutions adapt to technological change?",
]

rate_results = {}
for attr_name, attr_config in attributes.items():
    print(f"\n{'='*40}\nAttribute: {attr_name}\n{'='*40}")
    treated_scores, untreated_scores = [], []

    for prompt in sample_prompts:
        original = generate_response(client, prompt)
        rewrite_add = rewrite_text(client, original, attr_config["add"])
        rewrite_remove = rewrite_text(client, rewrite_add, attr_config["remove"])

        score_add = score_response(rm_model, rm_tokenizer, prompt, rewrite_add)
        score_remove = score_response(rm_model, rm_tokenizer, prompt, rewrite_remove)

        treated_scores.append(score_add)
        untreated_scores.append(score_remove)
        print(f"  {prompt[:50]}... | +{attr_name}={score_add:.3f}, -{attr_name}={score_remove:.3f}")

    ate = np.mean(treated_scores) - np.mean(untreated_scores)
    se = np.sqrt(np.var(treated_scores, ddof=1)/len(treated_scores) + np.var(untreated_scores, ddof=1)/len(untreated_scores))
    rate_results[attr_name] = {"ate": ate, "se": se}
    print(f"  ATE({attr_name}) = {ate:.4f} +/- {se:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
attrs = list(rate_results.keys())
ates = [rate_results[a]["ate"] for a in attrs]
ses = [rate_results[a]["se"] for a in attrs]
colors = ["#e74c3c" if a < 0 else "#2ecc71" for a in ates]
ax.barh(attrs, ates, xerr=ses, color=colors, alpha=0.8, capsize=5)
ax.set_xlabel("Average Treatment Effect on RM Score")
ax.set_title("RATE: What Does the Reward Model Actually Reward?")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

## Section E: Discussion & Homework

### Key Takeaways

**The RLHF pipeline** (Christiano → InstructGPT → ChatGPT) transforms a pretrained LM into an aligned assistant through three steps: supervised fine-tuning on demonstrations, reward model training on human preferences, and PPO optimization against the RM. Each step introduces design choices with social consequences.

**Constitutional AI** shifts the question from "what do humans prefer?" to "what principles should govern AI behavior?" — and who gets to write those principles? The 16 critique templates from [Bai et al. (2022)](https://arxiv.org/abs/2212.08073) encode specific normative commitments about harm, bias, and safety. These are institutional choices made explicit.

**Reward model explainability** (RATE) reveals that RMs often reward attributes we didn't intend — verbosity, flattery, particular writing styles. If we can't explain what the RM rewards, how confident are we that the aligned model serves the right objectives?

**For your research:** Think about who defines "preferred" in your institutional context. In healthcare, is a preferred AI response one that a doctor would give, or one a patient would want to hear? In education, do we optimize for satisfaction or learning outcomes? The reward function *is* the institution's values — made explicit and differentiable.

### Homework

1. **Train the reward model** on the full `ultrafeedback_binarized` dataset (not just 5K) and compare accuracy.

2. **Run RATE** on an attribute relevant to your memo research question (e.g., "empathy" for healthcare, "rigor" for education, "neutrality" for governance).

3. **Reflect:** What did the RM learn that surprised you? What attributes would you *want* a reward model to be sensitive to in your domain?

Share your findings, reflections, and results.

---
# Module 3: GRPO, DeepSeek-R1 & Societies of Thought

PPO (Module 1, Section G) powers RLHF — but it requires a learned critic $V(s)$, which adds complexity and instability. **GRPO** (Group Relative Policy Optimization) simplifies PPO by dropping the critic entirely. Instead of learning a value baseline, GRPO samples a group of completions for each prompt and uses their relative performance as the baseline. DeepSeek-R1 showed this is enough to teach models to reason.

The key result of [DeepSeek-R1 (2025)](https://arxiv.org/abs/2501.12948): a model trained with pure RL (no supervised fine-tuning on reasoning traces) spontaneously develops chain-of-thought reasoning, self-verification, and "aha moments" where it corrects its own errors. This is emergent behavior — not taught, but learned through reward optimization.

[Kim et al. (2025)](https://arxiv.org/) provide a framework for understanding *why* this works: reasoning models don't just generate longer text — they simulate internal "societies of thought" where different perspectives question, conflict with, and reconcile each other. This is computational deliberation, and it mirrors how diverse teams solve problems better than individuals.

In this module we train a model with GRPO on math problems and analyze the reasoning traces that emerge.

---

## Section A: GRPO — Group Relative Policy Optimization

PPO's clipped surrogate objective (Module 1, Section G) uses an advantage $\hat{A}_t = G_t - V_\phi(s_t)$ computed from a learned critic. GRPO replaces this with a **group-based advantage**:

1. For each prompt $x$, sample $G$ completions: $y_1, \ldots, y_G \sim \pi_\theta(y|x)$
2. Score each with a reward function: $r_1, \ldots, r_G$
3. Compute advantages using group statistics: $\hat{A}_i = \frac{r_i - \text{mean}(r_1, \ldots, r_G)}{\text{std}(r_1, \ldots, r_G)}$
4. Update policy with the same clipped objective as PPO

No critic, no value function, no bootstrapping — just relative comparisons within a group. This is simpler, more stable, and scales better. [DeepSeek-R1](https://arxiv.org/abs/2501.12948) used GRPO to train a 671B parameter model that matches OpenAI's o1 on math and coding benchmarks.

We use TRL's `GRPOTrainer` on `Qwen/Qwen2.5-0.5B-Instruct` with math problems from `AI-MO/NuminaMath-TIR`. The reward functions check (1) mathematical accuracy and (2) whether the model uses a `<think>` tag for reasoning.

For more: [TRL GRPOTrainer docs](https://huggingface.co/docs/trl/grpo_trainer), [DeepSeek-R1 paper](https://arxiv.org/abs/2501.12948), [Mini-R1 tutorial](https://www.philschmid.de/mini-deepseek-r1).

In [ ]:
import re
import torch
from datasets import load_dataset

grpo_dataset = load_dataset("AI-MO/NuminaMath-TIR", split="train[:2000]")

def extract_answer(text):
    boxed = re.findall(r"\\boxed\{([^}]*)\}", text)
    if boxed:
        return boxed[-1].strip()
    nums = re.findall(r"[-+]?\d*\.?\d+", text.split("\n")[-1] if text.strip() else "")
    return nums[-1] if nums else ""

def accuracy_reward(completions, **kwargs):
    solutions = kwargs.get("solution", [])
    rewards = []
    for completion, solution in zip(completions, solutions):
        pred = extract_answer(completion[0]["content"] if isinstance(completion, list) else completion)
        true = extract_answer(solution)
        rewards.append(1.0 if pred == true else 0.0)
    return rewards

def format_reward(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        has_think = "<think>" in text and "</think>" in text
        rewards.append(1.0 if has_think else 0.0)
    return rewards

SYSTEM_PROMPT = "You are a helpful math assistant. Show your reasoning inside <think>...</think> tags, then give your final answer."

def make_prompt(example):
    return {"prompt": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["problem"]},
    ]}

grpo_dataset = grpo_dataset.map(make_prompt)
print(f"GRPO dataset: {len(grpo_dataset)} problems")
print(f"Sample: {grpo_dataset[0]['prompt'][-1]['content'][:100]}")

In [ ]:
from trl import GRPOTrainer, GRPOConfig

training_args = GRPOConfig(
    learning_rate=2e-5,
    max_steps=100,
    per_device_train_batch_size=4,
    max_completion_length=256,
    num_generations=4,
    optim="adamw_torch",
    bf16=True,
    output_dir="grpo_qwen_math",
    logging_steps=10,
    log_completions=True,
    report_to="none",
)

grpo_trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=[accuracy_reward, format_reward],
    args=training_args,
    train_dataset=grpo_dataset,
)

grpo_trainer.train()

print("\nSample completions after training:")
test_problems_sample = grpo_dataset.select(range(3))
grpo_tokenizer = grpo_trainer.tokenizer
grpo_model = grpo_trainer.model
for ex in test_problems_sample:
    text = grpo_tokenizer.apply_chat_template(ex["prompt"], add_generation_prompt=True, tokenize=False)
    inputs = grpo_tokenizer(text, return_tensors="pt").to(grpo_model.device)
    with torch.no_grad():
        out = grpo_model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7)
    response = grpo_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nProblem: {ex['prompt'][-1]['content'][:100]}")
    print(f"Response: {response[:300]}")

## Section B: Analyzing Reasoning Traces — Societies of Thought

[Kim et al. (2025)](https://arxiv.org/) argue that reasoning models don't just produce longer outputs — they simulate internal **societies of thought**. The model generates traces that exhibit conversational patterns: questioning, perspective-shifting, conflict, and reconciliation. These are the same patterns that make diverse human teams effective at problem-solving.

We test this empirically. We generate traces from `DeepSeek-R1-Distill-Qwen-1.5B` (a 1.5B distillation of the full R1 model that retains its reasoning style) and compare them against `Qwen2.5-1.5B-Instruct` (a standard instruction-following model of the same size). We classify six behavioral patterns via regex:

| Pattern | Example markers |
|---------|----------------|
| Self-questioning | "let me think", "wait", "hmm" |
| Self-correction | "actually", "no, wait", "that's wrong" |
| Verification | "let me check", "verify", "to confirm" |
| Perspective shift | "but what if", "alternatively", "another way" |
| Conflict | "however", "this contradicts", "doesn't match" |
| Reconciliation | "therefore", "so combining", "the answer is" |

If the "societies of thought" hypothesis holds, the reasoning model should show significantly more of these patterns — especially self-correction and perspective shifts — than the base model.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
import matplotlib.pyplot as plt

reasoning_model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

reasoning_tokenizer = AutoTokenizer.from_pretrained(reasoning_model_name)
reasoning_model = AutoModelForCausalLM.from_pretrained(
    reasoning_model_name, torch_dtype=torch.bfloat16, device_map="auto"
)

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, torch_dtype=torch.bfloat16, device_map="auto"
)

test_problems = [
    "What is 23 * 17?",
    "If a train travels 120 miles in 2 hours, and then 90 miles in 1.5 hours, what is its average speed?",
    "A bag has 3 red and 5 blue marbles. You draw 2 without replacement. What is P(both red)?",
    "Solve: 2x + 5 = 3x - 7",
    "What is the sum of the first 20 positive integers?",
    "If f(x) = x^2 + 3x - 4, what are the roots?",
    "A rectangle has perimeter 36 and length twice its width. What is the area?",
    "How many ways can you arrange the letters in MISSISSIPPI?",
]

def generate_trace(model, tokenizer, problem, max_new_tokens=512):
    messages = [{"role": "user", "content": problem}]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.6)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("Generating reasoning model traces...")
reasoning_traces = [generate_trace(reasoning_model, reasoning_tokenizer, p) for p in test_problems]

print("Generating base model traces...")
base_traces = [generate_trace(base_model, base_tokenizer, p) for p in test_problems]

behavior_patterns = {
    "self_questioning": r"(let me think|wait|hmm|let's see|consider)",
    "self_correction": r"(actually|no,? wait|that's wrong|let me reconsider|I made a mistake)",
    "verification": r"(let me check|verify|double.check|to confirm|indeed)",
    "perspective_shift": r"(but what if|alternatively|on the other hand|another way|from a different)",
    "conflict": r"(however|but this contradicts|this doesn't match|inconsistent)",
    "reconciliation": r"(therefore|so combining|putting it together|in conclusion|the answer is)",
}

reasoning_counts = {k: 0 for k in behavior_patterns}
base_counts = {k: 0 for k in behavior_patterns}

for trace in reasoning_traces:
    for behavior, pattern in behavior_patterns.items():
        reasoning_counts[behavior] += len(re.findall(pattern, trace.lower()))

for trace in base_traces:
    for behavior, pattern in behavior_patterns.items():
        base_counts[behavior] += len(re.findall(pattern, trace.lower()))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(behavior_patterns))
width = 0.35
ax.bar(x - width/2, [reasoning_counts[k] for k in behavior_patterns], width, label="Reasoning Model", color="#3498db")
ax.bar(x + width/2, [base_counts[k] for k in behavior_patterns], width, label="Base Model", color="#e74c3c")
ax.set_xticks(x)
ax.set_xticklabels(behavior_patterns.keys(), rotation=30, ha="right")
ax.set_ylabel("Frequency")
ax.set_title("Societies of Thought: Reasoning Behaviors in Model Traces")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nExample reasoning trace ({test_problems[0]}):")
print(reasoning_traces[0][:500])
print(f"\nExample base trace ({test_problems[0]}):")
print(base_traces[0][:500])

## Section C: Discussion & Homework

### Key Takeaways

**GRPO** demonstrates that you don't need a learned critic to train reasoning — group-relative baselines are sufficient. This simplicity is why DeepSeek chose GRPO over PPO for training R1. The implication for alignment: simpler RL algorithms can produce sophisticated behaviors when the reward signal is clear.

**DeepSeek-R1** showed that reasoning emerges from pure RL without supervised examples of chain-of-thought. The model discovers self-verification, error correction, and "aha moments" on its own. [Zhang et al. (2025)](https://arxiv.org/abs/2501.12599) in "Interplay" further find that RL works best at the "edge of competence" — too easy and the model learns nothing, too hard and it can't get reward signal.

**Societies of Thought** gives a framework for *why* reasoning emerges: the model learns to simulate multiple perspectives and resolve conflicts between them. This mirrors social science findings on collective intelligence — diverse perspectives produce better decisions than any individual.

**For your research:** If you're studying how AI agents reason in institutional contexts, the SoT framework suggests that better reasoning comes not from smarter individuals but from richer internal deliberation. How does this apply to AI-assisted decision-making in your domain?

### Homework

1. **Run GRPO** for more training steps (200-500) and track how accuracy improves. Does the `<think>` format become more consistent?

2. **Analyze reasoning traces** on problems from your domain (e.g., social science reasoning, policy analysis, ethical dilemmas). Do the same SoT patterns appear?

3. **Reflect:** What kinds of reasoning patterns would you *want* an AI agent to exhibit in your institutional context? How might you design reward functions to encourage them?

Share your findings, reflections, and results.

---
# Module 4: Multi-Agent RL, Theory of Mind & Collaborative Rewards

So far our RL agents have operated alone — one agent, one environment, one reward. But most social systems involve multiple agents whose actions affect each other. The Prisoner's Dilemma is the canonical example: two agents must independently choose to cooperate or defect, and the outcome depends on both choices simultaneously. Individual rationality leads to mutual defection, but collective welfare demands cooperation.

Multi-agent RL (MARL) introduces two challenges that single-agent RL avoids: (1) *nonstationarity* — the environment changes because other agents are learning too, and (2) *credit assignment* — when a team succeeds, which agent contributed? [Cross et al. (2025)](https://arxiv.org/abs/2407.07086) address nonstationarity through Theory of Mind — maintaining hypotheses about what other agents are doing. [Bo et al. (2024)](https://arxiv.org/) address credit assignment through counterfactual rewards — asking "what would have happened without agent $i$?"

These are not just technical problems. They are the same problems faced by institutions: how do organizations coordinate agents with different incentives? How do you reward individual contribution in team outcomes? RL gives us a computational framework to study these questions.

---

## Section A: Iterated Prisoner's Dilemma with RL Agents

The [Prisoner's Dilemma](https://en.wikipedia.org/wiki/Prisoner%27s_dilemma) is the canonical multi-agent problem. Two agents simultaneously choose to **cooperate** (C) or **defect** (D). The payoff matrix:

|  | C | D |
|--|---|---|
| **C** | (3, 3) | (0, 5) |
| **D** | (5, 0) | (1, 1) |

Individual rationality says defect — it's dominant regardless of what the other does. But mutual cooperation (3, 3) beats mutual defection (1, 1). This is the core tension of institutional design: individual incentives vs. collective welfare.

In the **Iterated** Prisoner's Dilemma (IPD), agents play 100 rounds and can condition on history. This opens space for cooperation through reciprocity. We implement Q-learning agents with state = (my_last_action, opponent_last_action) and run three experiments:

1. **Individual reward** → defection dominates
2. **Collaborative reward** (team sum) → cooperation emerges
3. **Mixed reward** $r = \alpha \cdot r_\text{individual} + (1-\alpha) \cdot r_\text{team}$ → sweep $\alpha \in [0, 1]$

This directly parallels the reward design problem in RLHF (Module 2): the reward function encodes whose interests matter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

COOPERATE, DEFECT = 0, 1

def ipd_payoff(a1, a2):
    payoff_matrix = {
        (COOPERATE, COOPERATE): (3, 3),
        (COOPERATE, DEFECT): (0, 5),
        (DEFECT, COOPERATE): (5, 0),
        (DEFECT, DEFECT): (1, 1),
    }
    return payoff_matrix[(a1, a2)]

class QLearningAgent:
    def __init__(self, lr=0.1, gamma=0.95, epsilon=0.1):
        self.q = defaultdict(lambda: np.zeros(2))
        self.lr = lr
        self.gamma = gamma
        self.epsilon = epsilon

    def act(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(2)
        return int(np.argmax(self.q[state]))

    def update(self, state, action, reward, next_state):
        td_target = reward + self.gamma * np.max(self.q[next_state])
        self.q[state][action] += self.lr * (td_target - self.q[state][action])

def run_ipd(agent1, agent2, n_episodes=500, n_rounds=100, reward_type="individual"):
    coop_history, r1_history, r2_history = [], [], []
    for episode in range(n_episodes):
        s1, s2 = (0, 0), (0, 0)
        ep_coop, ep_r1, ep_r2 = 0, 0, 0
        for step in range(n_rounds):
            a1 = agent1.act(s1)
            a2 = agent2.act(s2)
            r1, r2 = ipd_payoff(a1, a2)
            if reward_type == "team":
                r1 = r2 = r1 + r2
            s1_new = (a1, a2)
            s2_new = (a2, a1)
            agent1.update(s1, a1, r1, s1_new)
            agent2.update(s2, a2, r2, s2_new)
            s1, s2 = s1_new, s2_new
            ep_coop += (a1 == 0) + (a2 == 0)
            ep_r1 += r1
            ep_r2 += r2
        coop_history.append(ep_coop / (2 * n_rounds))
        r1_history.append(ep_r1)
        r2_history.append(ep_r2)
    return coop_history, r1_history, r2_history

agent1_ind = QLearningAgent(); agent2_ind = QLearningAgent()
coop_ind, r1_ind, r2_ind = run_ipd(agent1_ind, agent2_ind, reward_type="individual")

agent1_team = QLearningAgent(); agent2_team = QLearningAgent()
coop_team, r1_team, r2_team = run_ipd(agent1_team, agent2_team, reward_type="team")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
window = 20
ax1.plot(np.convolve(coop_ind, np.ones(window)/window, mode="valid"), label="Individual Reward", color="#e74c3c")
ax1.plot(np.convolve(coop_team, np.ones(window)/window, mode="valid"), label="Team Reward", color="#2ecc71")
ax1.set_xlabel("Episode"); ax1.set_ylabel("Cooperation Rate")
ax1.set_title("IPD: Individual vs Team Reward")
ax1.legend(); ax1.set_ylim(-0.05, 1.05)

ax2.plot(np.convolve(r1_ind, np.ones(window)/window, mode="valid"), label="Individual", color="#e74c3c")
ax2.plot(np.convolve(r1_team, np.ones(window)/window, mode="valid"), label="Team", color="#2ecc71")
ax2.set_xlabel("Episode"); ax2.set_ylabel("Agent 1 Reward")
ax2.set_title("Reward Trajectories"); ax2.legend()
plt.tight_layout(); plt.show()

print(f"Individual reward → cooperation: {np.mean(coop_ind[-50:]):.1%}")
print(f"Team reward → cooperation: {np.mean(coop_team[-50:]):.1%}")

In [ ]:
alphas = np.linspace(0, 1, 11)
coop_rates_by_alpha = []

for alpha in alphas:
    agent1 = QLearningAgent(lr=0.1, epsilon=0.2)
    agent2 = QLearningAgent(lr=0.1, epsilon=0.2)
    coop_counts = 0
    total = 0
    for episode in range(200):
        s1, s2 = (0, 0), (0, 0)
        for step in range(100):
            a1 = agent1.act(s1)
            a2 = agent2.act(s2)
            r1_ind, r2_ind = ipd_payoff(a1, a2)
            r_team = r1_ind + r2_ind
            r1 = alpha * r1_ind + (1 - alpha) * r_team
            r2 = alpha * r2_ind + (1 - alpha) * r_team
            s1_new = (a1, a2)
            s2_new = (a2, a1)
            agent1.update(s1, a1, r1, s1_new)
            agent2.update(s2, a2, r2, s2_new)
            s1, s2 = s1_new, s2_new
            if episode >= 150:
                coop_counts += (a1 == 0) + (a2 == 0)
                total += 2
    coop_rates_by_alpha.append(coop_counts / total if total > 0 else 0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(alphas, coop_rates_by_alpha, "o-", color="#3498db", linewidth=2, markersize=8)
ax.set_xlabel(r"$\alpha$ (1 = pure individual, 0 = pure team)")
ax.set_ylabel("Cooperation Rate (last 50 episodes)")
ax.set_title(r"Cooperation vs Reward Mixing: $r = \alpha \cdot r_{ind} + (1-\alpha) \cdot r_{team}$")
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

print(f"Pure individual (alpha=1): {coop_rates_by_alpha[-1]:.1%} cooperation")
print(f"Pure team (alpha=0): {coop_rates_by_alpha[0]:.1%} cooperation")
print(f"50/50 mix (alpha=0.5): {coop_rates_by_alpha[5]:.1%} cooperation")

## Section B: Theory of Mind Module

[Cross et al. (2025)](https://arxiv.org/abs/2407.07086) introduce the *Hypothetical Minds* framework for multi-agent RL. The core idea: instead of treating other agents as part of the environment (which makes it nonstationary), maintain explicit **hypotheses** about what the other agent is doing and update them based on observed behavior.

This is computational Theory of Mind — the same cognitive capacity that lets humans predict others' intentions. The update rule is Rescorla-Wagner, borrowed from classical conditioning:

$$V(h) \leftarrow V(h) + \alpha \cdot \big(\text{accuracy}(h) - V(h)\big)$$

where $V(h)$ is the value of hypothesis $h$, and $\text{accuracy}(h)$ is how well $h$ predicted the opponent's last action. The agent acts based on its best hypothesis.

We apply this to the IPD from Section A. The ToM agent maintains hypotheses like "opponent plays tit-for-tat", "opponent always defects", "opponent cooperates if I cooperated last round". We compare against a vanilla Q-learner and show ToM adapts faster to strategy switches.

This is inspired by the architecture in [Cross et al. (2025)](https://arxiv.org/abs/2407.07086) — see the [Hypothetical-Minds repo](https://github.com/locross93/Hypothetical-Minds) for the full LLM-based implementation with natural-language hypotheses.

In [ ]:
class ToMAgent:
    def __init__(self, lr=0.3, threshold=0.7):
        self.hypotheses = {
            "tit_for_tat": lambda my_last, opp_last: opp_last if opp_last is not None else 0,
            "always_cooperate": lambda my_last, opp_last: 0,
            "always_defect": lambda my_last, opp_last: 1,
            "grim_trigger": None,
            "random": lambda my_last, opp_last: np.random.randint(2),
        }
        self.values = {h: 0.5 for h in self.hypotheses}
        self.lr = lr
        self.threshold = threshold
        self.grim_triggered = False
        self.hypotheses["grim_trigger"] = self._grim_trigger

    def _grim_trigger(self, my_last, opp_last):
        if opp_last == 1:
            self.grim_triggered = True
        return 1 if self.grim_triggered else 0

    def predict_opponent(self, my_last, opp_last):
        best_h = max(self.values, key=self.values.get)
        return self.hypotheses[best_h](my_last, opp_last)

    def update_hypotheses(self, my_last, opp_last, opp_actual):
        for h_name, h_func in self.hypotheses.items():
            predicted = h_func(my_last, opp_last)
            accuracy = 1.0 if predicted == opp_actual else 0.0
            self.values[h_name] += self.lr * (accuracy - self.values[h_name])

    def act(self, my_last, opp_last):
        predicted_opp = self.predict_opponent(my_last, opp_last)
        if predicted_opp == 0:
            return 0
        else:
            return 1

def run_ipd_tom_vs_switching(n_episodes=300, switch_every=75):
    tom = ToMAgent()
    strategies = ["always_cooperate", "always_defect", "tit_for_tat", "always_cooperate"]
    tom_rewards, opp_rewards, current_strategy = [], [], []

    for ep in range(n_episodes):
        strat_idx = min(ep // switch_every, len(strategies) - 1)
        strat_name = strategies[strat_idx]
        current_strategy.append(strat_name)
        ep_tom_r, ep_opp_r = 0, 0
        my_last, opp_last = None, None

        for step in range(100):
            a_tom = tom.act(my_last, opp_last)
            if strat_name == "always_cooperate":
                a_opp = 0
            elif strat_name == "always_defect":
                a_opp = 1
            elif strat_name == "tit_for_tat":
                a_opp = (my_last if my_last is not None else 0)
            else:
                a_opp = 0
            r_tom, r_opp = ipd_payoff(a_tom, a_opp)
            tom.update_hypotheses(my_last, opp_last, a_opp)
            my_last, opp_last = a_tom, a_opp
            ep_tom_r += r_tom
            ep_opp_r += r_opp
        tom_rewards.append(ep_tom_r)
        tom.grim_triggered = False

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    ax1.plot(tom_rewards, alpha=0.3, color="#3498db")
    window = 10
    ax1.plot(np.convolve(tom_rewards, np.ones(window)/window, mode="valid"), color="#3498db", linewidth=2, label="ToM Agent Reward")
    for i, s in enumerate(strategies):
        ax1.axvline(x=i*switch_every, color="red", linestyle="--", alpha=0.5)
        ax1.text(i*switch_every + 5, max(tom_rewards)*0.95, s, fontsize=8, color="red")
    ax1.set_ylabel("Episode Reward")
    ax1.set_title("ToM Agent vs Switching Opponent")
    ax1.legend()

    hypothesis_names = list(tom.values.keys())
    ax2.bar(hypothesis_names, [tom.values[h] for h in hypothesis_names], color="#2ecc71")
    ax2.set_ylabel("Hypothesis Value")
    ax2.set_title("Final ToM Hypothesis Values")
    ax2.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

run_ipd_tom_vs_switching()

## Section C: Credit Assignment — Counterfactual Rewards

When a team of agents succeeds, who gets the credit? Splitting reward equally ignores individual contributions. [Bo et al. (2024)](https://arxiv.org/) propose **counterfactual credit assignment** inspired by Shapley values:

$$\text{Credit}_i = R_\text{team} - R_{\text{team} \setminus i}$$

Remove agent $i$, compute the team reward without them, and the difference is their contribution. If removing an agent barely changes the outcome, they contributed little. If the team falls apart without them, they were essential.

This connects to a deep problem in institutional design: how do organizations reward individual contribution within team outcomes? Academic departments, firms, and governments all face this credit assignment problem. COPPER's counterfactual approach is one computational answer — compare to what would have happened in the counterfactual world where the agent didn't participate.

We apply counterfactual credit assignment to a multi-agent IPD with 4 agents and compare it against uniform credit splitting.

In [ ]:
def run_team_ipd(n_agents=4, n_episodes=500, n_rounds=50, use_counterfactual=True):
    agents = [QLearningAgent(lr=0.1, epsilon=0.15) for _ in range(n_agents)]
    cooperation_history = []
    credit_history = {i: [] for i in range(n_agents)}

    for episode in range(n_episodes):
        states = [(0, 0)] * n_agents
        ep_coop = 0
        for step in range(n_rounds):
            actions = [agent.act(state) for agent, state in zip(agents, states)]
            n_cooperators = sum(1 for a in actions if a == 0)
            team_reward = n_cooperators * 3 - (n_agents - n_cooperators) * 1

            for i, agent in enumerate(agents):
                if use_counterfactual:
                    actions_without_i = actions[:i] + actions[i+1:]
                    n_coop_without = sum(1 for a in actions_without_i if a == 0)
                    team_without = n_coop_without * 3 - (n_agents - 1 - n_coop_without) * 1
                    credit = team_reward - team_without
                    credit_history[i].append(credit)
                    reward = credit
                else:
                    reward = team_reward / n_agents

                opp_action = 0 if (n_cooperators - (1 if actions[i] == 0 else 0)) > n_agents // 2 else 1
                new_state = (actions[i], opp_action)
                agent.update(states[i], actions[i], reward, new_state)
                states[i] = new_state

            ep_coop += n_cooperators / n_agents
        cooperation_history.append(ep_coop / n_rounds)

    return cooperation_history, credit_history

coop_counterfactual, credits_cf = run_team_ipd(use_counterfactual=True)
coop_uniform, credits_uni = run_team_ipd(use_counterfactual=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
window = 20
ax1.plot(np.convolve(coop_counterfactual, np.ones(window)/window, mode="valid"),
         label="Counterfactual Credit", color="#2ecc71", linewidth=2)
ax1.plot(np.convolve(coop_uniform, np.ones(window)/window, mode="valid"),
         label="Uniform Credit", color="#e74c3c", linewidth=2)
ax1.set_xlabel("Episode")
ax1.set_ylabel("Cooperation Rate")
ax1.set_title("Counterfactual vs Uniform Credit Assignment")
ax1.legend()

agent_ids = list(credits_cf.keys())
mean_credits = [np.mean(credits_cf[i][-1000:]) for i in agent_ids]
ax2.bar(agent_ids, mean_credits, color="#3498db", alpha=0.8)
ax2.set_xlabel("Agent ID")
ax2.set_ylabel("Mean Counterfactual Credit")
ax2.set_title("Credit Distribution Across Agents")
ax2.axhline(y=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

print(f"Final cooperation rate — Counterfactual: {np.mean(coop_counterfactual[-50:]):.1%}")
print(f"Final cooperation rate — Uniform: {np.mean(coop_uniform[-50:]):.1%}")

## Section D: Discussion & Homework

### Key Takeaways

**The reward function encodes institutional values.** In the IPD, switching from individual to team reward changes the emergent behavior from defection to cooperation. The $\alpha$-sweep shows this is continuous — partial alignment of incentives yields partial cooperation. This is exactly the design problem facing real institutions: how much weight to give individual vs. collective outcomes.

**Theory of Mind enables adaptation.** The ToM agent outperforms Q-learners against switching opponents because it maintains explicit hypotheses about others' strategies. [Cross et al. (2025)](https://arxiv.org/abs/2407.07086) show this scales to complex environments with LLM-generated natural-language hypotheses. The implication: agents that model other agents' intentions coordinate better.

**Counterfactual credit sharpens incentives.** Uniform reward splits create free-rider problems. Counterfactual credit — asking "what would have happened without you?" — aligns individual incentives with actual contribution. [Bo et al. (2024)](https://arxiv.org/) show this improves both learning speed and final performance in multi-agent tasks.

**For your research:** Most social institutions are multi-agent systems with imperfect credit assignment. How does your institution of interest handle this? Could counterfactual reasoning improve it?

### Homework

1. **Extend the IPD** to N > 2 agents with different strategy mixes. How does cooperation scale with group size?

2. **Test ToM** against more complex strategy sequences. Can you design a strategy that fools the ToM agent?

3. **Apply counterfactual credit** to a collaborative task from your domain. How would you define "team reward" and "agent removal" in your context?

Share your findings, reflections, and results.

---
# Module 5: RL for AI Agents & Institutions — Final Homework

You have now built the full toolkit:
- **Module 1:** Foundations — from bandits to PPO, the algorithms that power everything
- **Module 2:** Alignment — RLHF trains models to follow human preferences, Constitutional AI encodes principles, RATE reveals what reward models actually learn
- **Module 3:** Reasoning — GRPO teaches models to think step-by-step, and reasoning traces exhibit "societies of thought"
- **Module 4:** Multi-agent — reward design determines cooperation vs. defection, Theory of Mind enables adaptation, counterfactual credit solves attribution

AI agents increasingly operate within institutions — healthcare systems, courts, markets, universities, governments. RL determines how they learn, what they optimize, and who they serve. The reward function is the institution's values made explicit and differentiable.

---

## Final Homework

### Part 1: Design (written, ~500 words)

Design an RL system for a social institution related to your memo research question. Specify:

- **Agent(s):** Who or what is learning? What are their action spaces?
- **Environment:** What is the state space? What does the agent observe?
- **Reward:** Human preferences (RLHF)? AI feedback (Constitutional AI)? Rule-based? Collaborative? Whose interests does it encode?
- **Alignment mechanism:** KL penalty? Constitutional principles? Process rewards? How do you prevent reward hacking?
- **Failure modes:** What could go wrong? Sycophancy? Misaligned incentives? Gaming the reward?

### Part 2: Implement (choose one)

Pick one implementation that connects to your design:

**(a) Train a small LM with RLHF or GRPO** — Adapt the code from Module 2 (Sections A-B) or Module 3 (Section A) to your domain. Use prompts and reward criteria relevant to your research question.

**(b) Build a multi-agent system** — Adapt the IPD from Module 4 (Section A) with a payoff matrix that models your institution. Add Theory of Mind (Section B) or counterfactual credit (Section C).

**(c) Analyze a reward model using RATE** — Adapt Module 2 (Section D) to test attributes relevant to your domain. What does the RM reward that it shouldn't? What does it miss?

**(d) Analyze reasoning traces** — Adapt Module 3 (Section B) to prompts from your domain. Do reasoning models exhibit "societies of thought" on social science questions?

### Part 3: Reflect (~300 words)

- What social scientific insights emerged from your implementation?
- How does RL reshape the institution you studied?
- What would you change about the reward function if you could redesign it?

Share your design document, code, outputs, and reflection.